In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [2]:
from IPython.display import Math, display
show = lambda func: display(Math(func.__str__()))

In [3]:
def readAirfoil(file='airfoil.dat'):
    with open(file, 'r') as f:
        data = f.read().split('\n\n')
    shape = [[], []]
    for d in range(2):
        shape[d].append([[float(n.strip(' ')) for n in thing.split('  ') if n] for thing in data[d].split('\n') if thing])
    df = pd.DataFrame({'xb': [lis[0] for lis in shape[0][0]], 'top': [lis[1] for lis in shape[0][0]], 'bot': [lis[1] for lis in shape[1][0]]})
    return df.set_index('xb')
readAirfoil().to_csv('aerofoil.csv')

In [4]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
lin = LinearRegression()
poly = PolynomialFeatures(degree=len(df))
lin.fit(poly.fit_transform(df.index.to_numpy().reshape(-1, 1)), df[['top', 'bot']].values)
df[['ptop', 'pbot']] = lin.predict(poly.transform(df.index.to_numpy().reshape(-1, 1)))

NameError: name 'df' is not defined

In [ ]:
from MathFunctions.Polynomial import Polynomial
top, bot = [Polynomial([lin.intercept_[p] + lin.coef_[p][0]] + list(lin.coef_[p][1:]), symbol='\\left(\\frac{x}{c}\\right)') for p in range(2)]
camber = (top + bot) / 2
camber.s = bot.s
df['camber'] = camber(df.index)
df.plot(figsize=(18, 4), y=['ptop', 'pbot', 'camber'])

In [ ]:
top.max(0.15, 0.4), top(0.15)
tmax, bmin, t15, b15 =  top.max(0.16, 0.4), bot.min(0.15, 0.4), top(0.15), bot(0.15)
tsk, bsk = (tmax + t15) / 2, (bmin + b15) / 2
coord = list(np.array([[0.15, bsk], [0.75, bsk], [0.75, tsk], [0.15, tsk], [0.15, bsk]]).T)
df.plot(figsize=(18, 4), y=['top', 'bot', 'camber'])
plt.plot(*coord)

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from numpy import linspace

dev = False # If you aren't Kaizad this should be disabled.

def PlotContour(xp, y, z, xtit, ytit, title, x):
    fig = go.Figure(data=go.Heatmap(
        x=xp,
        y=y,
        z=z,
        connectgaps=False,
        colorscale='Picnic',
        zsmooth='best'
    ), layout=go.Layout(
        xaxis=dict(title=xtit, autorange='reversed'),
        yaxis=dict(title=ytit),
        title=f'Cross Sectional {title} Stress Distribution at {x = } m'
    ))
    if dev:
        fig.write_html(f'/var/www/html/{title.lower()}.html')
    return fig

def NormalStressPlot(oxx, x, Ch, Ca, shape, tsk, **others):
    zi = list(linspace(Ch - Ca, Ch, 1200, dtype='float'))
    yi = list(linspace(-Ch, Ch, 600, dtype='float'))
    o_zy = [[None if abs(y_i) > shape(z_i) else oxx(z_i, y_i)(x)*1e-6 for z_i in zi] for y_i in yi]
    return PlotContour(zi, yi, o_zy,
        'Chordwise Distribution [m]',
        'Vertical Distribution [m]',
        'Normal', x)

def ShearStressPlot(oxx, x, Ch, Ca, shape, tsk, tsp, **others):
    zi = list(linspace(Ch - Ca, Ch, 1800, dtype='float'))
    yi = list(linspace(-Ch, Ch, 1200, dtype='float'))
    o_zy = [[None if abs(y_i) > shape(z_i) or (abs(y_i) < shape(z_i) - tsk and not -tsp < z_i < tsp) \
        else oxx(z_i, y_i)(x)*1e-6 for z_i in zi] for y_i in yi]
    return PlotContour(zi, yi, o_zy,
        'Chordwise Distribution [m]',
        'Vertical Distribution [m]',
        'Shear', x)

def InternalLoading(x0, x1, **Loads):
    titles = [k + f' [kN{"" if k[0].upper() == "V" else "m"}]' for k in Loads]
    fig = make_subplots(rows=len(list(Loads.keys())), cols=1, shared_xaxes=True, vertical_spacing=0.05)
    xs = linspace(x0, x1, 5000)
    for i, (li, load) in enumerate(zip(Loads.keys(), Loads.values())):
        fig.append_trace(go.Scatter(
            x=xs,
            y=[load(xi)*1e-3 for xi in xs],
            name=li,
        ), row=i+1, col=1)
        fig.update_yaxes(title_text=titles[i], row=i+1, col=1)

    fig.update_xaxes(title_text="Spanwise Position [meters]", row=len(Loads.keys()), col=1)
    fig.update_layout(
        title="Internal Load (NVM) Diagram",
        template="plotly_dark")

    if dev:
        fig.write_html('/var/www/html/nvmnum.html')
    return fig

def Deflections(x0, x1, pi, **defs):
    titles = [k + f' [{"millimeters" if k[0].lower() in ["w", "v"] else "degrees"}]' for k in defs]
    fig = make_subplots(rows=len(list(defs.keys())), cols=1, shared_xaxes=True, vertical_spacing=0.01)
    xs = linspace(x0, x1, 5000)
    for i, (li, load) in enumerate(zip(defs.keys(), defs.values())):
        fig.append_trace(go.Scatter(
            x=xs,
            y=[load(xi)*(180 / pi if 'degrees' in titles[i] else 1e3) for xi in xs],
            name=li,
        ), row=i+1, col=1)
        fig.update_yaxes(title_text=titles[i], row=i+1, col=1)

    fig.update_xaxes(title_text="Spanwise Position [meters]", row=len(defs.keys()), col=1)
    fig.update_layout(
        title="Deflection Diagram")

    if dev:
        fig.write_html('/var/www/html/deflectionsnum.html')
    return fig


In [8]:
from Analysis import Stringer, WingBox, WingStructure
from Equilibrium import PointLoad, Moment, RunningLoad, EquilibriumEquation, DistributedMoment
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from MathFunctions.Mechanics import StepFunction

class Engines:
    def __init__(self, ThrustHover, ThrustCruise, positions: list[float, int], weight):
        self.n, self.Thover, self.Tcruise = len(positions), ThrustHover, ThrustCruise
        self.pos = positions
        self.w = weight
    __repr__ = __str__ = lambda self: "Engines(" + ', '.join(f"{k}={self.__dict__[k]}" for k in self.__dict__) + ")"

class WingLoads:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)
        self.wing, self.acp = [None]*2
        self.Fx, self.Fy, self.Fz, self.Mx, self.My, self.Mz = [None]*6
        self.RFx, self.RFy, self.RFz, self.RMx, self.RMy, self.RMz = [None]*6
        
        self.box = WingBox(self.tsk, self.tsp, self.frac, self.toc) # Check 0.6
        self.box.StrPlacement(self.nStrT, self.nStrB, self.StrA)
        self.wing = WingStructure(self.span, self.taper, self.cr, self.box)
        self.acp = (self.xac - 0.45) * self.mac # Redefine
        
    def __setitem__(self, key, item):
        self.__dict__[key] = item

    __getitem__ = lambda self, key: self.__dict__[key]

    __repr__ = __str__ = lambda self: "Wingloads(" + ', '.join(f"{k}={self.__dict__[k]}" for k in self.__dict__) + ")"

    def equilibriumCruise(self, dragDistr, liftDistr, macDistr, wingWeight):
        acp = self.acp
        Thrusts = [PointLoad([-self.engines.Tcruise, -self.engines.w, 0], [-0.45*self.wing(p).b/self.frac, 0, p]) \
                   for p in self.engines.pos] # Redefine x, y
        Drag = RunningLoad([dragDistr[1], [0]*len(dragDistr[1])], dragDistr[0], axis=2, poa=(acp, 0))
        Lift = RunningLoad([[0]*len(liftDistr[1]), liftDistr[1]], liftDistr[0], axis=2, poa=(acp, 0))
        Mac = DistributedMoment(values=[[0, 0, mom] for mom in macDistr[1]], positions=macDistr[0])
        weights = RunningLoad(values=[[0]*2, [-wingWeight / self.span]*2], positions=[0, self.span/2], axis=2)
        eqn = EquilibriumEquation(kloads=[Lift, weights, Drag, Mac] + Thrusts,
                                 ukloads=[PointLoad([1, 0, 0], [0, 0, 0]), PointLoad([0, 1, 0], [0, 0, 0]),
                                          PointLoad([0, 0, 1], [0, 0, 0]), Moment([1, 0, 0]),
                                          Moment([0, 1, 0]), Moment([0, 0, 1])])
        print(eqn.SetupEquation())
        self.RFx, self.RFy, self.RFz, self.RMx, self.RMy, self.RMz = solved = eqn.SolveEquation()
        return solved
    
    def internalLoads(self, dragDistr, liftDistr, macDistr, wingWeight):
        if not any([self.RFx, self.RFy, self.RFz, self.RMx, self.RMy, self.RMz]):
            self.equilibriumCruise(dragDistr, liftDistr, macDistr, wingWeight)

        lin = LinearRegression()
        poly = PolynomialFeatures(degree=20)
        X = poly.fit_transform(dragDistr[0].reshape(-1, 1))
        lin.fit(X, np.array([dragDistr[1], liftDistr[1], macDistr[1]]).T)
        dragcoef, liftcoef, momcoef = lin.coef_
        interd, interl, interm = lin.intercept_
    
        drag = StepFunction([[dragcoef[i], 0, i] for i in range(len(dragcoef))]) + interd
        lift = StepFunction([[liftcoef[i], 0, i] for i in range(len(liftcoef))]) + interl
        Mac = StepFunction([[momcoef[i], 0, i] for i in range(len(momcoef))]) + interm
        Thrust = StepFunction([[-self.engines.Tcruise, p, 0] for p in self.engines.pos])
        ThrustW = StepFunction([[-self.engines.w, p, 0] for p in self.engines.pos])
        MThrust = StepFunction([[0.45*self.wing(p).b * self.engines.w / self.frac, p, 0] for p in self.engines.pos])
    
        wgt = StepFunction([[-wingWeight / self.span, 0, 0]])
        self.Vy = -(lift + wgt).integral(self.RFy) - ThrustW
        self.Vx = -(Thrust + drag.integral() + self.RFx)
        self.T = -(Mac + lift * self.acp).integral(self.RMz) - MThrust
        self.My, self.Mx = self.Vx.integral(self.RMy), self.Vy.integral(-self.RMx)
        return lift, wgt
        


In [9]:
from cg_est import *

b = (14 * 5.25) ** 0.5

args = dict(span=10, taper=0.4, cr=0.5, tsk=1e-3, tsp=1e-2,
            toc=0.17, nStrT=5, nStrB=3, StrA=1e-4, mac=0.65, xac=0.25,
            engines=Engines(2960, 2960, list(np.linspace(0.1*b/2, 0.8*b/2, 4)), 400 * 9.81 / 8),
           frac=0.6)

loads = WingLoads(**args)

config = 1
WoS = 1745
Pmax = 8.77182
mProp = 400 / 16
thickness = 3e-3
w_fus= 1.3; h_fus=1.6; l_fus=4
nmax = 3.2
mTO = 1800

w = Weight(95, Wing(mTO, 5.25, 5.25, 1.5*nmax, 14, [0.4, 3.6], config),
           Fuselage(mTO, Pmax, l_fus, 5, l_fus/2, config),
          LandingGear(mTO, l_fus/2),
          Propulsion(16, [mProp]*16, pos_prop=[3.6]*int(8) + [0.4]*int(8)),
          85, 3, 483.15, l_fus/2, [0.8, 1.3, 1.3, 2.5, 2.5])

wingWeight = w.wing.get_weight()[0] * 9.81
lift = nmax * 1.5 * w.mtom * 9.81
drag = lift / 19.03
pos = np.linspace(0, b/2, 10000)
dragd = 2 * drag / (np.pi * b) * np.sqrt(1 - 4 * np.power(pos / b, 2))
liftd = 2 * lift / (np.pi * b) * np.sqrt(1 - 4 * np.power(pos / b, 2))
Mac = 0.0645 * 0.5 * 1.2 * 53 ** 2 * 5.25 * 0.65

solved = loads.equilibriumCruise([pos, dragd], [pos, liftd], [pos, [Mac / b]*len(pos)], wingWeight)

lift, wgt = loads.internalLoads([pos, dragd], [pos, liftd], [pos, [Mac / b]*len(pos)], wingWeight)
InternalLoading(0, b/2, **{l: loads[l] for l in 'T, My, Mx, Vx, Vy'.split(', ')}).show(renderer='iframe')

(array([[1., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 1.]]), array([ 10334.9339733 , -24642.2220921 ,     -0.        ,  43229.46431427,
        20100.88835917,   3198.63473141]))


In [10]:
fig = make_subplots(rows=1, cols=1, shared_xaxes=True)
xs = pos
fig.append_trace(go.Scatter(
    x=xs,
    y=liftd,
    name='Lift',
), row=1, col=1)
fig.append_trace(go.Scatter(
    x=xs,
    y=[lift(k) for k in pos],
    name='Lift Approximation',
), row=1, col=1)

fig.append_trace(go.Scatter(
    x=xs,
    y=[wingWeight / b]*len(pos),
    name='Weight',
), row=1, col=1)

fig.update_yaxes(title_text='Lift [N]', row=1, col=1)

fig.update_xaxes(title_text="Spanwise Position [meters]", row=1, col=1)
fig.update_layout(
    title="Lift vs Weight",
    template="plotly_dark")
fig.show(renderer='iframe')

In [ ]:
equation = EquilibriumEquation(kloads=[RunningLoad([[0]*len(liftd), liftd], pos, axis=2, poa=(-0.2 * 0.65, 0)),
                                 RunningLoad(values=[[0]*2, [-wingWeight / b]*2], positions=[0, b/2], axis=2)],
                         ukloads=[PointLoad([0, 1, 0], [0, 0, 0]), Moment([1, 0, 0]), Moment([0, 0, 1])])
equation.SetupEquation()